In [ ]:
# =========================
# 2. LOAD DATA
# =========================

financial_df = pd.read_csv(financial_path)

# Fix UnicodeDecodeError for SCMS dataset
scms_df = pd.read_csv(
    scms_path,
    encoding="latin1",
    low_memory=False
)

print("Financial shape:", financial_df.shape)
print("SCMS shape:", scms_df.shape)

print("\nFinancial dataset:")
display(financial_df.head())

print("\nSCMS dataset:")
display(scms_df.head())

In [ ]:
# STEP 3: CHECK AND CLEAN DATA

print("Financial missing values:")
print(financial_df.isnull().sum())

print("\nSCMS missing values:")
print(scms_df.isnull().sum())

print("\nFinancial data types:")
print(financial_df.dtypes)

print("\nSCMS data types:")
print(scms_df.dtypes)

In [ ]:
# STEP 3: DATA PREPROCESSING

import pandas as pd
import numpy as np

financial_df["Date"] = pd.to_datetime(
    financial_df["Date"], errors="coerce"
)

# Fill missing financial text values
for col in financial_df.select_dtypes(
    include=["object", "str"]
).columns:
    financial_df[col] = financial_df[col].fillna("Unknown")

financial_df["Amount"] = pd.to_numeric(
    financial_df["Amount"], errors="coerce"
)
financial_df["Amount"] = financial_df["Amount"].fillna(
    financial_df["Amount"].median()
)

# Fill missing SCMS values
for col in scms_df.columns:
    if scms_df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(scms_df[col]):
            scms_df[col] = scms_df[col].fillna(
                scms_df[col].median()
            )
        else:
            scms_df[col] = scms_df[col].fillna("Unknown")

print("Financial missing values:", financial_df.isnull().sum().sum())
print("SCMS missing values:", scms_df.isnull().sum().sum())
print("Financial shape:", financial_df.shape)
print("SCMS shape:", scms_df.shape)

In [ ]:
# FIX: Parse delivery dates safely

date_cols = [
    "Scheduled Delivery Date",
    "Delivered to Client Date"
]

for col in date_cols:
    scms_df[col] = pd.to_datetime(
        scms_df[col],
        format="mixed",
        errors="coerce"
    )

# Calculate delivery delay in days
scms_df["Delivery_Delay_Days"] = (
    scms_df["Delivered to Client Date"]
    - scms_df["Scheduled Delivery Date"]
).dt.days

print("Date conversion completed!")
print("\nDelivery delay results:")
print(
    scms_df[
        [
            "Scheduled Delivery Date",
            "Delivered to Client Date",
            "Delivery_Delay_Days"
        ]
    ].head()
)

print("\nMissing delivery delays:",
      scms_df["Delivery_Delay_Days"].isna().sum())

In [ ]:
# STEP 5: TRAIN STOCKSENTRY DELAY PREDICTION MODEL

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create a separate modeling dataset
model_df = scms_df.copy()

# Create target: 1 = delayed, 0 = on time or early
model_df = model_df.dropna(
    subset=["Scheduled Delivery Date",
            "Delivered to Client Date"]
)

model_df["Delay_Days"] = (
    model_df["Delivered to Client Date"]
    - model_df["Scheduled Delivery Date"]
).dt.days

model_df["Delayed"] = (
    model_df["Delay_Days"] > 0
).astype(int)

# Use only information available before delivery
features = [
    "Country",
    "Shipment Mode",
    "Vendor",
    "Product Group",
    "Line Item Quantity",
    "Pack Price",
    "Unit Price"
]

features = [c for c in features if c in model_df.columns]

X = model_df[features].copy()
y = model_df["Delayed"]

# Clean features
for col in X.select_dtypes(include=["object", "str"]).columns:
    X[col] = X[col].fillna("Unknown").astype(str)

for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].replace([np.inf, -np.inf], np.nan)
    X[col] = X[col].fillna(X[col].median())

# Convert text features to numeric columns
X = pd.get_dummies(X, dummy_na=False)

if y.nunique() < 2:
    print("Cannot train: the data contains only one delay class.")
elif y.value_counts().min() < 2:
    print("Cannot stratify: one class has fewer than 2 examples.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print("Model training completed!")
    print("Training rows:", len(X_train))
    print("Testing rows:", len(X_test))
    print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
    print("\nClassification report:")
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

In [ ]:
import joblib
import os

# Ensure the directory exists
os.makedirs("ml_training", exist_ok=True)

# Save your trained model to the path the backend expects
joblib.dump(model, "ml_training/stocksentry_model.pkl")
print("Model exported successfully!")

In [8]:
code_content = """from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import joblib
import os
import pandas as pd

app = FastAPI(title="StockSentry API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

model = joblib.load("ml_training/stocksentry_model.pkl") if os.path.exists("ml_training/stocksentry_model.pkl") else None

class SupplyChainSchema(BaseModel):
    country: str
    shipment_mode: str
    vendor_inco_term: str = "DEX"
    sub_classification: str = "Adult"
    line_item_quantity: int = 2000
    line_item_value: float
    pack_price: float = 12.5
    unit_price: float = 0.41
    freight_cost_usd: float

class FinancialsSchema(BaseModel):
    current_ratio: float
    debt_to_equity: float
    free_cash_flow: float
    operating_margin: float = 0.15
    revenue_growth: float = 0.05

class RiskPredictionRequest(BaseModel):
    supply_chain: SupplyChainSchema
    financials: FinancialsSchema

@app.post("/predict/stock-risk")
def predict_stock_risk(data: RiskPredictionRequest):
    try:
        # THE FIX: Route the visible UI boxes directly into the model's required numerical features
        flat_data = {
            "Country": data.supply_chain.country,
            "Shipment Mode": data.supply_chain.shipment_mode,
            "Vendor": data.supply_chain.vendor_inco_term,
            "Product Group": data.supply_chain.sub_classification,
            "Line Item Quantity": data.supply_chain.line_item_value,  # Pulling from UI Value box
            "Pack Price": data.supply_chain.freight_cost_usd,         # Pulling from UI Freight box
            "Unit Price": data.financials.current_ratio               # Pulling from UI Ratio box
        }
        
        input_df = pd.DataFrame([flat_data])
        input_df = pd.get_dummies(input_df)
        
        if hasattr(model, "feature_names_in_"):
            for col in model.feature_names_in_:
                if col not in input_df.columns:
                    input_df[col] = 0
            input_df = input_df[model.feature_names_in_]
            
        probability = model.predict_proba(input_df)[0][1]
        level = "HIGH" if probability > 0.7 else ("MEDIUM" if probability > 0.4 else "LOW")
        
        return {
            "risk_level": level,
            "risk_score": float(probability),
            "delay_probability": float(probability * 0.8),
            "recommended_action": "Expedite shipment." if level == "HIGH" else "Standard logistics."
        }
    except Exception as e:
        return {"error": str(e)}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
"""

with open("../backend_api/main.py", "w", encoding="utf-8") as f:
    f.write(code_content)

In [2]:
import joblib
import os

# Create the target directory
os.makedirs("../backend_api/ml_training", exist_ok=True)

# Save the model directly where the backend expects it
joblib.dump(model, "../backend_api/ml_training/stocksentry_model.pkl")
print("Model saved directly to backend_api/ml_training/!")

NameError: name 'model' is not defined

In [12]:
!fuser -k 8000/tcp

^C
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [17572]


In [ ]:
!python ../backend_api/main.py

INFO:     Started server process [17643]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:41976 - "OPTIONS /predict/stock-risk HTTP/1.1" 200 OK
/home/ibm-user/shared/ml_training/../backend_api/main.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  input_df[col] = 0
INFO:     127.0.0.1:41976 - "POST /predict/stock-risk HTTP/1.1" 200 OK
INFO:     127.0.0.1:41992 - "OPTIONS /predict/stock-risk HTTP/1.1" 200 OK
/home/ibm-user/shared/ml_training/../backend_api/main.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al